In [1]:
import time
from pathlib import Path

import scipy.io as sio
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

from sysidentpy.neural_network import NARXNN
from sysidentpy.basis_function import Polynomial

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

data_path = Path('/Users/raab/Documents/Project_ML_CV-1/Benchmark_EEG_small/Benchmark_EEG_small.mat')
data = sio.loadmat(data_path)
u = data['data']['u'].item().flatten().reshape(-1, 1)
y = data['data']['y'].item().flatten().reshape(-1, 1)  # EEG output

print(f"Loaded dataset with {len(u)} samples")

Loaded dataset with 17920 samples


In [2]:
# Time-ordered split: train/val/test = 70/15/15
n = len(u)
n_train = int(n * 0.70)
n_val = int(n * 0.15)

u_train, u_val, u_test = u[:n_train], u[n_train:n_train + n_val], u[n_train + n_val:]
y_train, y_val, y_test = y[:n_train], y[n_train:n_train + n_val], y[n_train + n_val:]

scaler_u = StandardScaler()
scaler_y = StandardScaler()

u_train_s = scaler_u.fit_transform(u_train)
y_train_s = scaler_y.fit_transform(y_train)
u_val_s = scaler_u.transform(u_val)
y_val_s = scaler_y.transform(y_val)
u_test_s = scaler_u.transform(u_test)
y_test_s = scaler_y.transform(y_test)

print(f"Train: {len(u_train)} | Val: {len(u_val)} | Test: {len(u_test)}")

Train: 12544 | Val: 2688 | Test: 2688


In [3]:
class NARXNet(nn.Module):
    def __init__(self, input_size, hidden_1=64, hidden_2=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, hidden_1),
            nn.Tanh(),
            nn.Linear(hidden_1, hidden_2),
            nn.Tanh(),
            nn.Linear(hidden_2, 1),
        )

    def forward(self, x):
        return self.net(x)


def narx_regressor_count(ny, nu, degree=2):
    # Number of polynomial terms up to 'degree', excluding bias.
    m = ny + nu
    if degree != 2:
        raise ValueError("This helper currently supports degree=2.")
    linear_terms = m
    quadratic_terms = m * (m + 1) // 2
    return linear_terms + quadratic_terms


def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def estimate_mlp_flops_per_sample(input_size, hidden_1=64, hidden_2=32, output_size=1):
    # Approximate FLOPs for Linear layers only: 2 * in * out (mul + add).
    return (
        2 * input_size * hidden_1
        + 2 * hidden_1 * hidden_2
        + 2 * hidden_2 * output_size
    )

In [4]:
ny = 5
nu = 5
degree = 2

input_size = narx_regressor_count(ny=ny, nu=nu, degree=degree)
net = NARXNet(input_size=input_size, hidden_1=64, hidden_2=32)

model = NARXNN(
    net=net,
    ylag=ny,
    xlag=nu,
    basis_function=Polynomial(degree=degree),
    loss_func='mse_loss',
    optimizer='Adam',
    optim_params={},
    epochs=300,
    learning_rate=0.001,
    batch_size=32,
    verbose=False,
)

print(f"NARX input size: {input_size}")
print(f"Trainable params: {count_trainable_params(net)}")
print(f"Approx FLOPs/sample: {estimate_mlp_flops_per_sample(input_size):,}")

NARX input size: 65
Trainable params: 6337
Approx FLOPs/sample: 12,480


In [5]:
print("Training NARX...")
t0 = time.perf_counter()
model.fit(X=u_train_s, y=y_train_s)
train_time_s = time.perf_counter() - t0
print(f"Training completed in {train_time_s:.2f} s")

Training NARX...
Training completed in 33.01 s


In [6]:
# One-step prediction on validation and test
yhat_val_1_s = model.predict(X=u_val_s, y=y_val_s, steps_ahead=1)
yhat_test_1_s = model.predict(X=u_test_s, y=y_test_s, steps_ahead=1)

# Multi-step free-run simulation on validation and test
yhat_val_m_s = model.predict(X=u_val_s, y=y_val_s)
yhat_test_m_s = model.predict(X=u_test_s, y=y_test_s)

# Back to original scale
yhat_val_1 = scaler_y.inverse_transform(yhat_val_1_s)
yhat_test_1 = scaler_y.inverse_transform(yhat_test_1_s)
yhat_val_m = scaler_y.inverse_transform(yhat_val_m_s)
yhat_test_m = scaler_y.inverse_transform(yhat_test_m_s)

In [8]:
# Use multi-step prediction for VAF and align lengths safely.
y_pred_vaf = yhat_multistep.flatten()
y_true_vaf = ytest.flatten()[-len(y_pred_vaf):]

vaf = (1 - np.var(y_true_vaf - y_pred_vaf) / np.var(y_true_vaf)) * 100
print(f"VAF: {vaf:.2f}%")

VAF: -10.36%


In [7]:
def align_true_pred(y_true, y_pred):
    y_pred = np.asarray(y_pred).reshape(-1)
    y_true = np.asarray(y_true).reshape(-1)
    return y_true[-len(y_pred):], y_pred


def nrmse(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    denom = np.max(y_true) - np.min(y_true)
    return rmse / denom if denom > 0 else np.nan


def vaf(y_true, y_pred):
    denom = np.var(y_true)
    return (1 - np.var(y_true - y_pred) / denom) * 100 if denom > 0 else np.nan


def evaluate(y_true_full, y_pred):
    y_true_a, y_pred_a = align_true_pred(y_true_full, y_pred)
    mse = mean_squared_error(y_true_a, y_pred_a)
    rmse = np.sqrt(mse)
    return {
        "MSE": mse,
        "RMSE": rmse,
        "NRMSE": nrmse(y_true_a, y_pred_a),
        "VAF": vaf(y_true_a, y_pred_a),
    }

In [8]:
metrics = {
    "val_1step": evaluate(y_val, yhat_val_1),
    "val_multistep": evaluate(y_val, yhat_val_m),
    "test_1step": evaluate(y_test, yhat_test_1),
    "test_multistep": evaluate(y_test, yhat_test_m),
}

print("\n=== NARX Results ===")
for name, values in metrics.items():
    print(f"\n[{name}]")
    print(f"MSE   : {values['MSE']:.6f}")
    print(f"RMSE  : {values['RMSE']:.6f}")
    print(f"NRMSE : {values['NRMSE']:.4f} ({values['NRMSE'] * 100:.2f}%)")
    print(f"VAF   : {values['VAF']:.2f}%")

print("\n=== Complexity / Time ===")
print(f"Train time [s]     : {train_time_s:.2f}")
print(f"Trainable params   : {count_trainable_params(net)}")
print(f"Approx FLOPs/sample: {estimate_mlp_flops_per_sample(input_size):,}")


=== NARX Results ===

[val_1step]
MSE   : 0.146092
RMSE  : 0.382219
NRMSE : 0.0616 (6.16%)
VAF   : 84.96%

[val_multistep]
MSE   : 1.534215
RMSE  : 1.238634
NRMSE : 0.1997 (19.97%)
VAF   : -55.45%

[test_1step]
MSE   : 0.102112
RMSE  : 0.319549
NRMSE : 0.0490 (4.90%)
VAF   : 89.44%

[test_multistep]
MSE   : 1.262335
RMSE  : 1.123537
NRMSE : 0.1725 (17.25%)
VAF   : -27.38%

=== Complexity / Time ===
Train time [s]     : 33.01
Trainable params   : 6337
Approx FLOPs/sample: 12,480
